# Cicero R ⇄ `epione.tl` — function-by-function dictionary

For R users porting Cicero code to Python: every exported R function of cicero 1.3.9 (Monocle 3
branch), its parameters, the R call, the Python call on the **same input**, and the comparison
with R's output (R outputs dumped by `../r_reference_driver.R` and `../r_utils_driver.R`, which
together play the role of the kit's `r_per_function_dump.R`). Fixture: Cicero's `cicero_data`
(F1), seed 42. Orientation: R's CDS is peaks × cells, the AnnData here is cells × peaks.

## 1. Setup

In [1]:
import os, sys, json, time, subprocess, tempfile
for k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(k, "8")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yaml
import epione as epi
from epione.tl import _cicero as C
from epione.tl._rrng import RRandom
HERE = Path(".").resolve()                       # tests/parity/cicero/examples
PARITY = HERE.parent                             # tests/parity/cicero
sys.path.insert(0, str(PARITY))
import parity_check as P, parity_utils as U
D = Path(os.environ["EPIONE_CICERO_PARITY_DIR"])  # fixtures/, ref/, results/
manifest = yaml.safe_load((PARITY / "manifest.yaml").read_text())

ad, coords, genome = P.read_fixture(D / "fixtures/F1")
R1 = D / "ref/F1_seed42"; RU = D / "ref/F1_utils"
verdict = []
def add(fn, output, cls, metric, value, ok):
    shown = f"{value:.2e}" if isinstance(value, float) else str(value)
    verdict.append({"Function": fn, "Output": output, "Class": cls, "Metric": metric, "Value": shown, "Pass": "✅" if ok else "❌"})
    print(f"{fn} / {output}: {metric} = {value} -> {'✅' if ok else '❌'}")

## 2. Function by function

### 2.1 `make_atac_cds(input, binarize = FALSE)`

Builds the accessibility object from sparse `(peak, cell, count)` triplets; peaks sorted by chromosome and start.

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `input` | `input` | data.frame / path | — | 3 columns: peak, cell, count | sparse triplets; a path is read as tab-separated without header |
| `binarize` | `binarize` | logical | `FALSE` | TRUE/FALSE | counts > 0 become 1 (before duplicates are summed, as R) |

**R**
```r
input_cds <- make_atac_cds(cicero_data, binarize = TRUE)
```

In [2]:
cicero_data = pd.read_csv(RU / "cicero_data.tsv", sep="\t")
input_cds = epi.tl.make_atac_cds(cicero_data, binarize=True)
input_cds

AnnData object with n_obs × n_vars = 200 × 6146
    obs: 'cells', 'num_genes_expressed', 'Size_Factor'
    var: 'site_name', 'chr', 'bp1', 'bp2', 'num_cells_expressed'

**Comparison with R**

In [3]:
rows, cols, m = U.rmat(RU, "atac")
ok = list(input_cds.var_names) == rows and list(input_cds.obs_names) == cols and U.same_matrix(input_cds.X, m)
add("make_atac_cds", "matrix + order", "deterministic", "identical", ok, ok)

make_atac_cds / matrix + order: identical = True -> ✅


### 2.2 `make_cicero_cds(cds, reduced_coordinates, k = 50, ...)`

kNN aggregation of cells into overlapping bins (≤ 5000 candidates, ≥ 90 % overlap rejected), detect_genes, size factors, normalisation.

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `cds` | `adata` | cell_data_set | — | peaks × cells (R) / cells × peaks (Py) | accessibility |
| `reduced_coordinates` | `reduced_coordinates` | matrix / data.frame | — (Py: `"X_umap"`) | cells × d | embedding used for the kNN graph; Py also accepts an `obsm` key |
| `k` | `k` | integer | `50` | > 1 | cells per bin |
| `summary_stats` | `summary_stats` | character | `NULL` | numeric `pData`/`obs` columns | averaged per bin as `mean_<col>` |
| `size_factor_normalize` | `size_factor_normalize` | logical | `TRUE` | | divide bins by size factor |
| `silent` | `silent` | logical | `FALSE` | | overlap QC message |
| `return_agg_info` | `return_agg_info` | logical | `FALSE` | | also return the cell → bin table |
| — | `random_state` | int / `RRandom` / Generator | `0` | | **new in Python**: `s` ≡ `set.seed(s)` before the call; an `RRandom` continues a stream |
| — | `rng` | str | `"R"` | `"R"`, `"numpy"` | **new**: R's generator (exact) or numpy's (distributional) |
| — | `chosen` | int sequence | `None` | | **new**: inject the bins (0-based seed cells), e.g. from R |
| — | `layer` | str | `None` | | **new**: use `adata.layers[layer]` |

**R**
```r
set.seed(42)
cicero_cds <- make_cicero_cds(input_cds, reduced_coordinates = umap_coords)
```

In [4]:
cicero_cds = epi.tl.make_cicero_cds(ad, reduced_coordinates=coords, k=50, random_state=42, silent=True)
cicero_cds

AnnData object with n_obs × n_vars = 25 × 6146
    obs: 'agg_cell', 'num_genes_expressed', 'Size_Factor'
    var: 'chr', 'bp1', 'bp2', 'num_cells_expressed'
    uns: 'cicero'

**Comparison with R**

In [5]:
r_bins = (R1 / "agg_cells.txt").read_text().split()
ok1 = list(cicero_cds.obs_names) == r_bins
err = float(abs(cicero_cds.X - P.read_ref_cicero_x(R1)).max())
add("make_cicero_cds", "bins (RNG)", "deterministic", "identical", ok1, ok1)
add("make_cicero_cds", "normalised matrix", "deterministic", "max abs err", err, err <= 1e-8)

make_cicero_cds / bins (RNG): identical = True -> ✅
make_cicero_cds / normalised matrix: max abs err = 0.0 -> ✅


### 2.3 `estimate_distance_parameter(cds, window = 5e5, maxit = 100, s = 0.75, sample_num = 100, ...)`

Bisection per random window for the smallest penalty scale that leaves ≤ 5 % of > 250 kb pairs connected.

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `cds` | `cds` | cicero CDS | — | output of make_cicero_cds | |
| `window` | `window` | numeric | `500000` | > distance_constraint | window size (bp) |
| `maxit` | `maxit` | integer | `100` | | bisection iterations |
| `s` | `s` | numeric | `0.75` | (0, 1) | power-law exponent |
| `sample_num` | `sample_num` | integer | `100` | | windows to calibrate on |
| `distance_constraint` | `distance_constraint` | integer | `250000` | < window | "long" pair distance |
| `distance_parameter_convergence` | `distance_parameter_convergence` | numeric | `1e-22` | | bisection step tolerance |
| `max_elements` | `max_elements` | integer | `200` | | skip denser windows |
| `genomic_coords` | `genomic_coords` | data.frame / path | `human.hg19.genome` (Py: required) | chr, length | chromosome lengths |
| `max_sample_windows` | `max_sample_windows` | integer | `500` | | windows tried at most |
| — | `random_state`, `rng` | | `0`, `"R"` | | **new**: as in make_cicero_cds |
| — | `sample_windows` | int sequence | `None` | 1-based | **new**: inject the sampled windows |
| — | `return_details` | bool | `False` | | **new**: per-window table |
| — | `n_jobs` | int | `1` | | **new**: threads (identical result) |

**R**
```r
set.seed(42)
distance_parameters <- estimate_distance_parameter(cicero_cds, genomic_coords = sample_genome)
```

In [6]:
dps = epi.tl.estimate_distance_parameter(cicero_cds, genomic_coords=genome, random_state=42, n_jobs=8)
dps[:5]

array([0.62980377, 0.68748428, 0.48090508, 0.39015473, 0.62980377])

**Comparison with R**

In [7]:
r_dps = np.fromfile(R1 / "distance_parameters.f64")
rel = float(np.max(np.abs(dps - r_dps) / r_dps))
add("estimate_distance_parameter", "per-window values", "deterministic-bounded", "max rel err", rel, rel <= 1e-6)

estimate_distance_parameter / per-window values: max rel err = 2.2730370368284633e-15 -> ✅


### 2.4 `generate_cicero_models(cds, distance_parameter, s = 0.75, window = 5e5, max_elements = 200, genomic_coords)`

Graphical lasso (glasso 1.11, ported line by line) in every overlapping window with the distance-scaled penalty.

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `cds` | `cds` | cicero CDS | — | | |
| `distance_parameter` | `distance_parameter` | numeric | — | > 0 | usually mean of estimate_distance_parameter |
| `s` | `s` | numeric | `0.75` | (0, 1) | |
| `window` | `window` | numeric | `500000` | | |
| `max_elements` | `max_elements` | integer | `200` | | |
| `genomic_coords` | `genomic_coords` | data.frame / path | `human.hg19.genome` (Py: required) | | |
| — | `n_jobs` | int | `1` | | **new**: threads (identical result) |
| — | `keep_wi` | bool | `True` | | **new**: keep the inverse (memory) |

**R**
```r
cicero_out <- generate_cicero_models(cicero_cds, distance_parameter = mean(distance_parameters), genomic_coords = sample_genome)
```

In [8]:
mean_dp = C._r_mean(dps)          # R's mean(): long-double sum + correction pass
models = epi.tl.generate_cicero_models(cicero_cds, mean_dp, genomic_coords=genome, n_jobs=8)
len(models)

313

**Comparison with R**

In [9]:
mt = pd.read_csv(R1 / "models.tsv", sep="\t")
st = ["glasso" if isinstance(m, C.CiceroModel) else m for m in models.values()]
ok = list(models) == mt.window.tolist() and st == mt.status.tolist()
add("generate_cicero_models", "window status", "deterministic", "identical", ok, ok)

generate_cicero_models / window status: identical = True -> ✅


### 2.5 `assemble_connections(cicero_model_list, silent = FALSE)`

cov2cor of every window model; pairs from overlapping windows reconciled (mean if same sign, else NA).

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `cicero_model_list` | `cicero_model_list` | list | — | output of generate_cicero_models | |
| `silent` | `silent` | logical | `FALSE` | | |

**R**
```r
conns <- assemble_connections(cicero_out)
```

In [10]:
conns = epi.tl.assemble_connections(models, silent=True)
conns.head()

,Peak1,Peak2,coaccess
0,chr18_10006196_10006822,chr18_9755702_9755970,-0.077102
1,chr18_10006196_10006822,chr18_9756925_9757590,0.000000
2,chr18_10006196_10006822,chr18_9771216_9771842,0.018597
3,chr18_10006196_10006822,chr18_9781976_9782901,-0.084068
4,chr18_10006196_10006822,chr18_9784605_9785105,-0.007434


**Comparison with R**

In [11]:
cmp = P.compare_conns(conns, P.read_ref_conns(R1))
add("assemble_connections", "coaccess", "deterministic", "max abs err", cmp["max_abs_err"], cmp["max_abs_err"] <= 1e-8 and cmp["only_py"] == cmp["only_r"] == 0)
add("assemble_connections", "row order / NA pattern", "deterministic", "identical", cmp["same_row_order"] and cmp["na_pattern_mismatch"] == 0, cmp["same_row_order"] and cmp["na_pattern_mismatch"] == 0)

assemble_connections / coaccess: max abs err = 8.049116928532385e-16 -> ✅
assemble_connections / row order / NA pattern: identical = True -> ✅


### 2.6 `run_cicero(cds, genomic_coords, window = 5e5, silent = FALSE, sample_num = 100)`

Wrapper: estimate_distance_parameter (maxit 100, distance_constraint 250000, convergence 1e-22) → mean → generate_cicero_models → assemble_connections.

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `cds` | `cds` | cicero CDS | — | | |
| `genomic_coords` | `genomic_coords` | data.frame / path | — | | |
| `window` | `window` | numeric | `500000` | | |
| `silent` | `silent` | logical | `FALSE` | | |
| `sample_num` | `sample_num` | integer | `100` | | |
| — | `random_state`, `rng`, `n_jobs` | | `0`, `"R"`, `1` | | **new** |

**R**
```r
set.seed(42)
conns <- run_cicero(cicero_cds, sample_genome)
```

In [12]:
conns_rc = epi.tl.run_cicero(cicero_cds, genome, random_state=42, n_jobs=8, silent=True)
conns_rc.attrs["mean_distance_parameter"]

0.6404942304943019

**Comparison with R**

In [13]:
r_mean = float(np.fromfile(R1 / "mean_distance_parameter.f64")[0])
cmp = P.compare_conns(conns_rc, P.read_ref_conns(R1))
add("run_cicero", "mean distance parameter", "deterministic", "rel err", abs(conns_rc.attrs["mean_distance_parameter"] - r_mean) / r_mean, conns_rc.attrs["mean_distance_parameter"] == r_mean)
add("run_cicero", "coaccess", "deterministic", "max abs err", cmp["max_abs_err"], cmp["max_abs_err"] <= 1e-8)

run_cicero / mean distance parameter: rel err = 0.0 -> ✅
run_cicero / coaccess: max abs err = 8.049116928532385e-16 -> ✅


### 2.7 `generate_ccans(connections_df, coaccess_cutoff_override = NULL, tolerance_digits = 2)`

Louvain modules (> 2 peaks) of the graph of pairs above a cutoff found by bisection on the CCAN count.

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `connections_df` | `connections_df` | data.frame | — | Peak1, Peak2, coaccess | |
| `coaccess_cutoff_override` | `coaccess_cutoff_override` | numeric | `NULL` | [0, 1] | skip the cutoff search |
| `tolerance_digits` | `tolerance_digits` | numeric | `2` | | cutoff precision |
| — | `random_state` | int / `RRandom` | `0` | | **new**: `s` ≡ `set.seed(s)` before the R call |
| — | `rng` | str | `"R"` | `"R"`, `"igraph"` | **new**: igraph Louvain on R's stream (exact) or python-igraph (distributional) |
| — | `silent` | bool | `False` | | **new** |

**R**
```r
set.seed(42)
ccan_assigns <- generate_ccans(conns)
```

In [14]:
rc = P.read_ref_conns(R1)
ccans = epi.tl.generate_ccans(rc, random_state=42, silent=True)
ccans.attrs["coaccess_cutoff"], ccans.CCAN.nunique()

(0.46, 131)

**Comparison with R**

In [15]:
from sklearn.metrics import adjusted_rand_score
cc = pd.read_csv(R1 / "ccan_cutoffs.tsv", sep="\t")
add("generate_ccans", "cutoff", "deterministic-bounded", "|Δ|", abs(ccans.attrs["coaccess_cutoff"] - cc.cutoff.iloc[0]), abs(ccans.attrs["coaccess_cutoff"] - cc.cutoff.iloc[0]) <= 0.02)
rr = {s: pd.read_csv(R1 / f"ccan_fixedcut_seed{s}.tsv", sep="\t") for s in cc.seed}
py = epi.tl.generate_ccans(rc, coaccess_cutoff_override=cc.cutoff.iloc[0], random_state=42, silent=True)
uni = sorted(set(py.Peak).union(*[set(v.Peak) for v in rr.values()]))
lab = lambda d: pd.Series(d.CCAN.values, index=d.Peak).reindex(uni).fillna(0).astype(int)
aris = {s: adjusted_rand_score(lab(v), lab(py)) for s, v in rr.items()}
print("ARI Py vs R seeds:", {k: round(v, 3) for k, v in aris.items()})
add("generate_ccans", "membership vs R seed 42", "clustering", "ARI", float(aris[42]), aris[42] >= 0.95)
r42 = pd.read_csv(R1 / "ccan_seed42.tsv", sep="\t")
same = len(ccans) == len(r42) and (ccans.Peak.values == r42.Peak.values).all() and (ccans.CCAN.values == r42.CCAN.values).all()
add("generate_ccans", "table, same seed (42)", "clustering", "identical", bool(same), bool(same))

generate_ccans / cutoff: |Δ| = 0.0 -> ✅


ARI Py vs R seeds: {42: 1.0, 2017: 0.935, 7: 0.956, 11: 0.952, 23: 0.935}
generate_ccans / membership vs R seed 42: ARI = 1.0 -> ✅
generate_ccans / table, same seed (42): identical = True -> ✅


### 2.8 `find_overlapping_ccans(ccan_assignments, min_overlap = 1)`

Pairs of CCANs whose genomic spans overlap (reciprocal).

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `ccan_assignments` | `ccan_assignments` | data.frame | — | Peak, CCAN | |
| `min_overlap` | `min_overlap` | integer | `1` | | bp |

**R**
```r
olap <- find_overlapping_ccans(ccan_assigns)
```

In [16]:
r_cc = pd.read_csv(R1 / "ccan_seed42.tsv", sep="\t")
ov = epi.tl.find_overlapping_ccans(r_cc)
ov.head()

,CCAN1,CCAN2
0,1,4
1,1,3
2,3,4
3,3,1
4,4,325


**Comparison with R**

In [17]:
r_ov = pd.read_csv(RU / "foccans.tsv", sep="\t")
ok = len(ov) == len(r_ov) and (ov.values == r_ov.values).all()
add("find_overlapping_ccans", "pairs (R's CCANs)", "deterministic", "identical", ok, ok)

find_overlapping_ccans / pairs (R's CCANs): identical = True -> ✅


### 2.9 `annotate_cds_by_site(cds, feature_data, verbose = FALSE, maxgap = 0, all = FALSE, header = FALSE)`

Adds overlapping features (e.g. promoter gene names) to the peak table; largest overlap wins.

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `cds` | `adata` | CDS | — | | |
| `feature_data` | `feature_data` | data.frame / path | — | chr, start, end, extra | 1-based closed |
| `verbose` | `verbose` | logical | `FALSE` | | |
| `maxgap` | `maxgap` | numeric / "nearest" | `0` | ≥ 0 | "nearest" not ported |
| `all` | `all` | logical | `FALSE` | | collapse all hits with "," |
| `header` | `header` | logical | `FALSE` | | for files |
| — | `inplace` | bool | `True` | | **new** |

**R**
```r
input_cds <- annotate_cds_by_site(input_cds, gene_annotation_sub)
```

In [18]:
prom = pd.read_csv(D / "fixtures/F1/promoters.tsv", sep="\t")
ad_ann = ad.copy(); epi.tl.annotate_cds_by_site(ad_ann, prom)
ad_ann.var.dropna().head(3)

,overlap,gene
chr18_10603_11103,1.0,AP005530.1
chr18_49557_50057,1.0,RP11-683L23.1
chr18_50240_50740,1.0,RP11-683L23.1


**Comparison with R**

In [19]:
ra = pd.read_csv(R1 / "annotated_sites.tsv", sep="\t", keep_default_na=False, na_values=["NA"]).set_index("peak").loc[ad_ann.var_names]
ok = (ad_ann.var.gene.fillna("").astype(str).values == ra.gene.fillna("").astype(str).values).all()
add("annotate_cds_by_site", "gene column", "deterministic", "identical", bool(ok), bool(ok))

annotate_cds_by_site / gene column: identical = True -> ✅


### 2.10 `build_gene_activity_matrix(input_cds, cicero_cons_info, site_weights = NULL, dist_thresh = 250000, coaccess_cutoff = 0.25)`

Promoter accessibility plus co-accessibility-weighted distal accessibility.

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `input_cds` | `adata` | CDS | — | `gene` column | |
| `cicero_cons_info` | `cicero_cons_info` | data.frame | — | Peak1, Peak2, coaccess | |
| `site_weights` | `site_weights` | numeric | `NULL` (all 1) | per peak | |
| `dist_thresh` | `dist_thresh` | numeric | `250000` | | |
| `coaccess_cutoff` | `coaccess_cutoff` | numeric | `0.25` | | |
| — | `gene_col`, `layer` | str | `"gene"`, `None` | | **new** |

**R**
```r
unnorm_ga <- build_gene_activity_matrix(input_cds, conns)
```

In [20]:
ga = epi.tl.build_gene_activity_matrix(ad_ann, rc)
ga

AnnData object with n_obs × n_vars = 200 × 602

**Comparison with R**

In [21]:
genes = (R1 / "unnorm_ga_genes.txt").read_text().split(); cells = (R1 / "unnorm_ga_cells.txt").read_text().split()
Rm = np.fromfile(R1 / "unnorm_ga.f64").reshape(len(genes), len(cells), order="F")
Pm = pd.DataFrame(ga.X.T.toarray(), index=ga.var_names, columns=ga.obs_names).loc[genes, cells].to_numpy()
err = float(np.abs(Pm - Rm).max()); add("build_gene_activity_matrix", "activity", "deterministic", "max abs err", err, err <= 1e-8)

build_gene_activity_matrix / activity: max abs err = 8.881784197001252e-16 -> ✅


### 2.11 `normalize_gene_activities(activity_matrices, cell_num_genes)`

Size-factor model log(total activity) ~ log(accessible sites), exp transform, column scaling.

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `activity_matrices` | `activity_matrices` | matrix / list | — | no all-zero genes or cells | |
| `cell_num_genes` | `cell_num_genes` | named numeric | — | per cell | accessible sites per cell |

**R**
```r
cicero_gene_activities <- normalize_gene_activities(unnorm_ga, num_genes)
```

In [22]:
kg = np.asarray(ga.X.sum(0)).ravel() != 0; kc = np.asarray(ga.X.sum(1)).ravel() != 0
gn = epi.tl.normalize_gene_activities(ga[kc][:, kg].copy(), pd.Series(np.asarray((ad.X > 0).sum(1)).ravel(), index=ad.obs_names))
gn

AnnData object with n_obs × n_vars = 200 × 602

**Comparison with R**

In [23]:
genes = (R1 / "ga_genes.txt").read_text().split(); cells = (R1 / "ga_cells.txt").read_text().split()
Rm = np.fromfile(R1 / "ga.f64").reshape(len(genes), len(cells), order="F")
Pm = pd.DataFrame(gn.X.T.toarray(), index=gn.var_names, columns=gn.obs_names).loc[genes, cells].to_numpy()
err = float(np.abs(Pm - Rm).max() / np.abs(Rm).max()); add("normalize_gene_activities", "activity", "deterministic", "max err / max|R|", err, err <= 1e-8)

normalize_gene_activities / activity: max err / max|R| = 6.661338147750939e-16 -> ✅


### 2.12 `aggregate_nearby_peaks(cds, distance = 1000)`

Sum peaks closer than `distance` (GenomicRanges::reduce).

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `cds` | `adata` | CDS | — | | |
| `distance` | `distance` | numeric | `1000` | | min.gapwidth |
| — | `layer` | str | `None` | | **new** |

**R**
```r
agg_cds <- aggregate_nearby_peaks(input_cds, distance = 10000)
```

In [24]:
agg = epi.tl.aggregate_nearby_peaks(input_cds, distance=10000)
agg

AnnData object with n_obs × n_vars = 200 × 1690
    obs: 'cells', 'num_genes_expressed', 'Size_Factor'
    var: 'site_name'

**Comparison with R**

In [25]:
import contextlib, io
if 'ures' not in dir():
    with contextlib.redirect_stdout(io.StringIO()):
        ures = U.main([str(RU), str(Path(tempfile.mkdtemp()) / 'u.json')])
g = ures['gates']['20']; add('aggregate_nearby_peaks', 'gate 20 (all cases of r_utils_driver.R)', 'deterministic', 'identical', g['pass'], g['pass'])

aggregate_nearby_peaks / gate 20 (all cases of r_utils_driver.R): identical = True -> ✅


### 2.13 `aggregate_by_cell_bin(cds, group_col)`

Sum cells per group of a `pData` column.

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `cds` | `adata` | CDS | — | | |
| `group_col` | `group_col` | character | — | `obs` column | |
| — | `layer` | str | `None` | | **new** |

**R**
```r
binned <- aggregate_by_cell_bin(input_cds, "timepoint")
```

In [26]:
cell_data = pd.read_csv(RU / 'cell_data.tsv', sep='\t').set_index('cell')
x = input_cds.copy(); x.obs['timepoint'] = cell_data.loc[x.obs_names, 'timepoint'].to_numpy()
binned = epi.tl.aggregate_by_cell_bin(x, 'timepoint')
binned.obs

,timepoint,num_genes_expressed,Size_Factor,n
T0,T0,2639,0.673041,51
T1,T1,3465,0.890082,47
T2,T2,3746,1.039088,55
T3,T3,4693,1.606483,47


**Comparison with R**

In [27]:
import contextlib, io
if 'ures' not in dir():
    with contextlib.redirect_stdout(io.StringIO()):
        ures = U.main([str(RU), str(Path(tempfile.mkdtemp()) / 'u.json')])
g = ures['gates']['21']; add('aggregate_by_cell_bin', 'gate 21 (all cases of r_utils_driver.R)', 'deterministic', 'identical', g['pass'], g['pass'])

aggregate_by_cell_bin / gate 21 (all cases of r_utils_driver.R): identical = True -> ✅


### 2.14 `compare_connections(conns1, conns2, maxgap = 0)`

Is each pair of `conns1` present (either orientation, within maxgap) in `conns2`?

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `conns1` | `conns1` | data.frame | — | | |
| `conns2` | `conns2` | data.frame | — | | |
| `maxgap` | `maxgap` | numeric | `0` | | bp |

**R**
```r
in_alt <- compare_connections(conns, alt, maxgap = 1000)
```

In [28]:
c1 = pd.read_csv(RU / 'cc_conns1.tsv', sep='\t'); alt = pd.read_csv(RU / 'cc_alt.tsv', sep='\t')
in_alt = epi.tl.compare_connections(c1, alt, maxgap=1000)
in_alt.sum(), len(in_alt)

(np.int64(365), 10866)

**Comparison with R**

In [29]:
import contextlib, io
if 'ures' not in dir():
    with contextlib.redirect_stdout(io.StringIO()):
        ures = U.main([str(RU), str(Path(tempfile.mkdtemp()) / 'u.json')])
g = ures['gates']['22']; add('compare_connections', 'gate 22 (all cases of r_utils_driver.R)', 'deterministic', 'identical', g['pass'], g['pass'])

compare_connections / gate 22 (all cases of r_utils_driver.R): identical = True -> ✅


### 2.15 `find_overlapping_coordinates(coord_list, coord, maxgap = 0)`

Peaks overlapping one or several coordinates (NA → `None`).

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `coord_list` | `coord_list` | character | — | | |
| `coord` | `coord` | character | — | one or many | commas allowed |
| `maxgap` | `maxgap` | numeric | `0` | | |

**R**
```r
find_overlapping_coordinates(peaks, "chr18:10,100-1246234")
```

In [30]:
epi.tl.find_overlapping_coordinates(list(input_cds.var_names), 'chr18:10,100-1246234')[:5]

['chr18_10025_10225',
 'chr18_10603_11103',
 'chr18_11604_13986',
 'chr18_49557_50057',
 'chr18_50240_50740']

**Comparison with R**

In [31]:
import contextlib, io
if 'ures' not in dir():
    with contextlib.redirect_stdout(io.StringIO()):
        ures = U.main([str(RU), str(Path(tempfile.mkdtemp()) / 'u.json')])
g = ures['gates']['23']; add('find_overlapping_coordinates', 'gate 23 (all cases of r_utils_driver.R)', 'deterministic', 'identical', g['pass'], g['pass'])

find_overlapping_coordinates / gate 23 (all cases of r_utils_driver.R): identical = True -> ✅


### 2.16 `df_for_coords(coord_strings)`

Coordinate strings to chr / bp1 / bp2 / Peak.

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `coord_strings` | `coord_strings` | character | — | `:`, `-`, `_` separators | |

**R**
```r
df_for_coords(c("chr1:2,039-30,239", "chrX:28884:101293"))
```

In [32]:
epi.tl.df_for_coords(['chr1:2,039-30,239', 'chrX:28884:101293'])

,chr,bp1,bp2,Peak
0,chr1,2039.0,30239.0,chr1:2039-30239
1,chrX,28884.0,101293.0,chrX:28884:101293


**Comparison with R**

In [33]:
import contextlib, io
if 'ures' not in dir():
    with contextlib.redirect_stdout(io.StringIO()):
        ures = U.main([str(RU), str(Path(tempfile.mkdtemp()) / 'u.json')])
g = ures['gates']['24']; add('df_for_coords', 'gate 24 (all cases of r_utils_driver.R)', 'deterministic', 'identical', g['pass'], g['pass'])

df_for_coords / gate 24 (all cases of r_utils_driver.R): identical = True -> ✅


### 2.17 `glasso::glasso(s, rho, nobs, zero, thr = 1e-4, maxit = 1e4, approx = FALSE, penalize.diagonal = TRUE, start, w.init, wi.init, trace)`

The solver Cicero calls (CRAN glasso 1.11, Fortran), ported subroutine by subroutine; cold start, exact path only.

**Parameters**

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `s` | `s` | matrix | — | square | covariance |
| `rho` | `rho` | scalar / vector / matrix | — | ≥ 0 | penalty |
| `nobs` | — | integer | `NULL` | | not ported (log-likelihood only) |
| `zero` | — | matrix | `NULL` | | not ported (Cicero does not use it) |
| `thr` | `thr` | numeric | `1e-4` | | |
| `maxit` | `maxit` | integer | `1e4` | | |
| `approx` | — | logical | `FALSE` | | not ported (Meinshausen–Bühlmann) |
| `penalize.diagonal` | `penalize_diagonal` | logical | `TRUE` | | |
| `start`, `w.init`, `wi.init` | — | | `"cold"` | | warm start not ported |
| `trace` | — | logical | `FALSE` | | not ported |

**R**
```r
GL <- glasso::glasso(cov_mat, rho_mat)
```

In [34]:
tab = pd.read_csv(R1 / "probes.tsv", sep="\t")
S, Rh, W, WI = (np.fromfile(R1 / f"probe_{m}.f64") for m in ("S", "rho", "w", "wi"))
errs = []
for _, r in tab.iterrows():
    n, o = int(r.n), int(r.offset); sl = slice(o, o + n * n)
    g = epi.tl.glasso(S[sl].reshape(n, n, order="F"), Rh[sl].reshape(n, n, order="F"))
    errs.append((np.abs(g.w - W[sl].reshape(n, n, order="F")).max(), np.abs(g.wi - WI[sl].reshape(n, n, order="F")).max(), g.niter == r.niter))
errs = np.array(errs, dtype=float)

**Comparison with R**

In [35]:
add("glasso", "w (40 Cicero windows)", "deterministic", "max abs err", float(errs[:, 0].max()), errs[:, 0].max() <= 1e-8)
add("glasso", "wi (40 Cicero windows)", "deterministic", "max abs err", float(errs[:, 1].max()), errs[:, 1].max() <= 1e-8)
add("glasso", "iterations", "deterministic", "identical", bool(errs[:, 2].all()), bool(errs[:, 2].all()))

glasso / w (40 Cicero windows): max abs err = 2.6645352591003757e-15 -> ✅
glasso / wi (40 Cicero windows): max abs err = 5.40421751793474e-09 -> ✅
glasso / iterations: identical = True -> ✅


### Not ported

| R function | Reason |
|---|---|
| `plot_connections`, `plot_accessibility_in_pseudotime` | Gviz / ggplot2 plotting; epione has its own track plots (`epi.pl.plot_browser_track`, `epi.pl.igv_browser`) |
| `ranges_for_coords` | returns a Bioconductor `GRanges`; `df_for_coords` gives the same coordinates as a DataFrame |

## 3. Aggregate verdict

In [36]:
pd.DataFrame(verdict)

,Function,Output,Class,Metric,Value,Pass
0,make_atac_cds,matrix + order,deterministic,identical,True,✅
1,make_cicero_cds,bins (RNG),deterministic,identical,True,✅
2,make_cicero_cds,normalised matrix,deterministic,max abs err,0.00e+00,✅
3,estimate_distance_parameter,per-window values,deterministic-bounded,max rel err,2.27e-15,✅
4,generate_cicero_models,window status,deterministic,identical,True,✅
5,assemble_connections,coaccess,deterministic,max abs err,8.05e-16,✅
6,assemble_connections,row order / NA pattern,deterministic,identical,True,✅
7,run_cicero,mean distance parameter,deterministic,rel err,0.00e+00,✅
8,run_cicero,coaccess,deterministic,max abs err,8.05e-16,✅
9,generate_ccans,cutoff,deterministic-bounded,|Δ|,0.00e+00,✅
